# Lesson 2: Probability and Bayesian Updating
### Conditional Probability, Natural Frequencies, Monty Hall & Sequential Learning
*Bayesian Cognitive Science & Data Analysis (2026)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/02_probability_bayesian_updating.ipynb)

---

## 🎯 Before Class
- **One-minute goal**: Master conditional probability, natural frequencies, and exact Bayesian updating in code without confusing likelihood with posterior density.
- **Prerequisites**: Lesson 1 concepts (estimands vs. estimates, uncertainty vs. variation), basic Python array operations.
- **Expected runtime**: ~90 minutes (all exercises run interactively on a standard, free Google Colab CPU).

## 💾 Make Your Copy
- **Google Colab**: Click `File` → `Save a copy in Drive` to edit and execute your own copy.
- **Local Jupyter**: Click `File` → `Download` → `Download .ipynb` and place in your course repository.

## 📋 Data Sources & Ethics
- **UC Berkeley Graduate Admissions (1973)**: Public aggregate contingency table (4,526 applications across 6 major departments; Bickel, Hammel, & O'Connell, 1975, *Science*; McElreath, *Statistical Rethinking* Ch. 11). Contains zero personal or identifiable records.
- **ROS Elections Economy**: Gelman, Hill, & Vehtari (2020), *Regression and Other Stories*, Ch. 9. Aggregate historical forecast and survey constants ($n=400$, $y=190$).
- **ROS Death Penalty Series**: Gelman et al. (2020), *Regression and Other Stories*, Ch. 4. Aggregate national poll proportions across decades.

## 🏆 Learning Targets
1. **Understand Conditional Probability & Denominators**: Calculate joint, marginal, and conditional probabilities from contingency tables; explain why reversing the condition changes the denominator reference set.
2. **Resolve the False Positive Paradox**: Simulate a disease screening protocol and natural frequency breakdown demonstrating why false positives dominate in low-prevalence regimes.
3. **Simulate the Monty Hall Paradox**: Calculate exact Bayesian matrix updates under the host's behavioral policy and verify the $2/3$ switching probability via 10,000-trial Monte Carlo simulation.
4. **Observe the Likelihood vs. Density Duality**: Contrast the forward generative probability mass function (sums to 1.0 over data) with the inverse inferential likelihood function (does not sum to 1.0 over parameters).
5. **Trace Sequential Bayesian Updating**: Implement conjugate Beta-Binomial updating step-by-step and verify order invariance.

## 📐 Cell Conventions
- `▶ RUN TOGETHER`: Instructor-paced walkthrough.
- `✍ WRITE`: Student response in Markdown.
- `🛑 STOP`: Class synchronization checkpoint.
- `🧪 CHANGE ONE THING`: Paired or independent modification.
- `✅ CHECK`: Automated invariant or visual diagnostic.
- `🏠 OPTIONAL HOMEWORK`: Non-graded extension.

### Classwork 0: Predict Before Running
`✍ WRITE`: Before executing any code, write your quick intuitive answers:
1. **Disease Screening**: If a rare disease affects 1 in 1,000 people and a test is 95% sensitive with a 5% false positive rate, what is the probability that a person who tests positive actually has the disease? (*Pick one: ~95%? ~50%? < 5%?*)
2. **Monty Hall**: You choose Door 1. The host opens Door 2 revealing a goat. Does switching to Door 3 double your chance of winning (from 1/3 to 2/3), or does it remain 50/50?
3. **Updating Order**: If you observe 5 coin flips $[H, T, H, H, T]$, does updating your belief sequentially after each flip give a different posterior distribution than updating once on all 5 flips simultaneously?

In [1]:
# ==============================================================================
# ▶ RUN TOGETHER: Environment Setup & Package Imports
# ==============================================================================
import sys
import os
import urllib.request
import datetime
import numpy as np
import pandas as pd
import scipy.stats as stats
import scipy.integrate as integrate
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("⚡ Running in Google Colab environment.")
    pio.renderers.default = "colab"
else:
    print("💻 Running in local environment.")

# Set random seed for reproducibility
np.random.seed(42)

print("Environment ready!")
print(f"Python: {sys.version.split()[0]} | NumPy: {np.__version__} | SciPy: {stats.__version__ if hasattr(stats, '__version__') else 'installed'} | Pandas: {pd.__version__}")

💻 Running in local environment.
Environment ready!
Python: 3.12.13 | NumPy: 2.4.3 | SciPy: installed | Pandas: 3.0.2


---
# Part I: Classwork (Guided Laboratory)

## ▶ RUN TOGETHER: 1. Conditional Probability & The Denominator Rule
In Bayesian conditioning, **the condition defines the reference group (the denominator)**:
$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)} = \frac{\text{Count}(A \text{ and } B)}{\text{Count}(B)}
$$
Reversing the condition changes the reference set:
$$
P(B \mid A) = \frac{P(A \cap B)}{P(A)} = \frac{\text{Count}(A \text{ and } B)}{\text{Count}(A)}
$$
Therefore, $P(A \mid B) \neq P(B \mid A)$ whenever $P(A) \neq P(B)$. In the 1973 UC Berkeley graduate admissions data, confusion between $P(\text{admit} \mid \text{female})$ and $P(\text{female} \mid \text{admit})$ illustrated this fundamental distinction.

In [2]:
# ==============================================================================
# Auto-fetch 1973 Berkeley Admissions Dataset from GitHub
# Source: Bickel, Hammel, & O'Connell (1975) Science; McElreath (Statistical Rethinking Ch. 11)
# ==============================================================================
DATA_DIR = "data"
DATA_FILE = "ucbadmit.csv"
LOCAL_PATH = os.path.join(DATA_DIR, DATA_FILE)
RAW_URL = f"https://raw.githubusercontent.com/iknyazeva/bayes-cogsci-book/main/data/{DATA_FILE}"

# Check for existing local file in possible locations
candidate_paths = [
    LOCAL_PATH,
    os.path.join("..", "..", DATA_DIR, DATA_FILE),
    os.path.join("..", DATA_DIR, DATA_FILE),
]
found_path = None
for p in candidate_paths:
    if os.path.exists(p):
        found_path = p
        break

# In Colab or if missing locally, download directly from GitHub
if found_path is None or IS_COLAB:
    os.makedirs(DATA_DIR, exist_ok=True)
    if not os.path.exists(LOCAL_PATH):
        print(f"📥 Downloading {DATA_FILE} from GitHub ({RAW_URL})...")
        urllib.request.urlretrieve(RAW_URL, LOCAL_PATH)
        print(f"✅ Successfully downloaded to {LOCAL_PATH}")
    found_path = LOCAL_PATH

df_ucb = pd.read_csv(found_path)
print(f"📂 Dataset loaded from {found_path}: {len(df_ucb)} rows")
display(df_ucb)

total_apps = df_ucb['applications'].sum()
total_admitted = df_ucb['admit'].sum()
female_apps = df_ucb[df_ucb['gender'] == 'female']['applications'].sum()
female_admitted = df_ucb[df_ucb['gender'] == 'female']['admit'].sum()
male_apps = df_ucb[df_ucb['gender'] == 'male']['applications'].sum()
male_admitted = df_ucb[df_ucb['gender'] == 'male']['admit'].sum()

# Conditional Probabilities
p_admit_given_female = female_admitted / female_apps
p_admit_given_male = male_admitted / male_apps
p_female_given_admit = female_admitted / total_admitted


print("=== 1973 Berkeley Admissions: Conditioning Breakdown ===")
print(f"Total Applications: {total_apps:,} | Total Admitted: {total_admitted:,}")
print(f"P(Admit | Female) = {female_admitted} / {female_apps} = {p_admit_given_female:.4f} ({p_admit_given_female*100:.1f}%)")
print(f"P(Admit | Male)   = {male_admitted} / {male_apps} = {p_admit_given_male:.4f} ({p_admit_given_male*100:.1f}%)")
print(f"P(Female | Admit) = {female_admitted} / {total_admitted} = {p_female_given_admit:.4f} ({p_female_given_admit*100:.1f}%)")

# Department A Specific Rates
dept_a = df_ucb[df_ucb['dept'] == 'A']
dept_a_male = dept_a[dept_a['gender'] == 'male'].iloc[0]
dept_a_female = dept_a[dept_a['gender'] == 'female'].iloc[0]

p_admit_dept_a_male = dept_a_male['admit'] / dept_a_male['applications']
p_admit_dept_a_female = dept_a_female['admit'] / dept_a_female['applications']

print("\n=== Department A Specific Conditioning ===")
print(f"P(Admit | Male, Dept A)   = {dept_a_male['admit']} / {dept_a_male['applications']} = {p_admit_dept_a_male:.4f} ({p_admit_dept_a_male*100:.1f}%)")
print(f"P(Admit | Female, Dept A) = {dept_a_female['admit']} / {dept_a_female['applications']} = {p_admit_dept_a_female:.4f} ({p_admit_dept_a_female*100:.1f}%)")

📂 Dataset loaded from ../../data/ucbadmit.csv: 12 rows


,dept,gender,admit,reject,applications
0,A,male,512,313,825
1,A,female,89,19,108
2,B,male,353,207,560
3,B,female,17,8,25
4,C,male,120,205,325
5,C,female,202,391,593
6,D,male,138,279,417
7,D,female,131,244,375
8,E,male,53,138,191
9,E,female,94,299,393


=== 1973 Berkeley Admissions: Conditioning Breakdown ===
Total Applications: 4,526 | Total Admitted: 1,755
P(Admit | Female) = 557 / 1835 = 0.3035 (30.4%)
P(Admit | Male)   = 1198 / 2691 = 0.4452 (44.5%)
P(Female | Admit) = 557 / 1755 = 0.3174 (31.7%)

=== Department A Specific Conditioning ===
P(Admit | Male, Dept A)   = 512 / 825 = 0.6206 (62.1%)
P(Admit | Female, Dept A) = 89 / 108 = 0.8241 (82.4%)


## 🛑 STOP 1: Check your understanding of conditioning
`✍ WRITE`:
1. Why does $P(\text{admit} \mid \text{female}) \neq P(\text{female} \mid \text{admit})$? State the exact denominator for both quantities.
2. In Department A, female applicants were admitted at a substantially higher rate than male applicants (82.4% vs 62.1%), yet in the aggregate table female applicants had a lower admission rate (30.4% vs 44.5%). Explain why aggregate conditioning across departments creates this reversal (Simpson's Paradox).

## ▶ RUN TOGETHER: 2. The False Positive Paradox & Natural Frequencies

When screening for a condition with low prior prevalence, false positives from the vast healthy population easily outnumber true positives from the small infected population.

By Bayes' Theorem:
$$
P(C \mid +) = \frac{P(+ \mid C) \, P(C)}{P(+ \mid C) \, P(C) + P(+ \mid \neg C) \, P(\neg C)}
$$

**Natural Frequency Representation (10,000 People)**:
Presenting probabilities as expected counts in a standard cohort makes Bayesian reasoning intuitive (Gigerenzer & Hoffrage, 1995):
- Prevalence $P(C) = 0.1\%$ (10 in 10,000)
- Sensitivity $P(+ \mid C) = 95\%$
- False positive rate $P(+ \mid \neg C) = 5\%$

In [3]:
def bayesian_screening(prevalence, sensitivity=0.95, false_positive_rate=0.05):
    """Compute posterior probability P(Condition | Positive Test)."""
    prior_c = prevalence
    prior_not_c = 1.0 - prevalence
    p_pos_given_c = sensitivity
    p_pos_given_not_c = false_positive_rate
    
    numerator = p_pos_given_c * prior_c
    denominator = numerator + (p_pos_given_not_c * prior_not_c)
    return numerator / denominator

# Natural frequency breakdown for a population of N = 10,000
N_pop = 10_000
prev = 0.001
sens = 0.95
fpr = 0.05

sick = N_pop * prev
healthy = N_pop * (1.0 - prev)

true_pos = sick * sens
false_neg = sick * (1.0 - sens)
false_pos = healthy * fpr
true_neg = healthy * (1.0 - fpr)

total_pos = true_pos + false_pos
post_prob = true_pos / total_pos

print(f"=== Natural Frequency Breakdown (N = {N_pop:,}) ===")
print(f"Truly Sick: {sick:.1f} | Truly Healthy: {healthy:.1f}")
print(f"├─ True Positives (+ | Sick):       {true_pos:.1f}")
print(f"├─ False Negatives (- | Sick):      {false_neg:.1f}")
print(f"├─ False Positives (+ | Healthy):   {false_pos:.1f}")
print(f"└─ True Negatives (- | Healthy):    {true_neg:.1f}")
print(f"\nTotal Positive Tests: {total_pos:.1f}")
print(f"P(Sick | Positive Test) = {true_pos:.1f} / {total_pos:.1f} = {post_prob:.4f} ({post_prob*100:.2f}%)")

=== Natural Frequency Breakdown (N = 10,000) ===
Truly Sick: 10.0 | Truly Healthy: 9990.0
├─ True Positives (+ | Sick):       9.5
├─ False Negatives (- | Sick):      0.5
├─ False Positives (+ | Healthy):   499.5
└─ True Negatives (- | Healthy):    9490.5

Total Positive Tests: 509.0
P(Sick | Positive Test) = 9.5 / 509.0 = 0.0187 (1.87%)


In [4]:
# Plotting the Prevalence Curve: How Prior Base Rate Dictates Posterior Confidence
prev_grid = np.linspace(0.0001, 0.30, 300)
post_grid = [bayesian_screening(p) * 100 for p in prev_grid]

fig_prev = go.Figure()
fig_prev.add_trace(go.Scatter(
    x=prev_grid * 100, y=post_grid, mode='lines',
    line=dict(color='#2563eb', width=2.5),
    name='P(Condition | + Test)'
))

# Highlight the 0.1% base rate and 10% base rate
p_01 = bayesian_screening(0.001) * 100
p_10 = bayesian_screening(0.10) * 100
fig_prev.add_trace(go.Scatter(
    x=[0.1, 10.0], y=[p_01, p_10], mode='markers+text',
    marker=dict(color=['#dc2626', '#16a34a'], size=10),
    text=[f"Base Rate 0.1% → {p_01:.1f}%", f"Base Rate 10% → {p_10:.1f}%"],
    textposition="top left", name='Key Benchmarks'
))

fig_prev.update_layout(
    title='The False Positive Paradox: Posterior Confidence vs. Base Rate (Sens=95%, FPR=5%)',
    xaxis_title='Prior Prevalence / Base Rate (%)',
    yaxis_title='Posterior Probability P(Condition | +) (%)',
    template='plotly_white', height=420
)
fig_prev.show()

## 🧪 CHANGE ONE THING: Adjust Test Specificity
`✍ WRITE`:
If a laboratory improves test specificity so the false positive rate drops from $5\%$ ($0.05$) to $0.1\%$ ($0.001$), what happens to $P(C \mid +)$ when prevalence is $0.1\%$?
Run `bayesian_screening(0.001, sensitivity=0.95, false_positive_rate=0.001)` and interpret why specificity matters so much more than sensitivity in low-base-rate settings.

## ▶ RUN TOGETHER: 3. The Monty Hall Paradox: Exact Bayes & Monte Carlo Simulation

In the Monty Hall problem:
1. You choose Door 1.
2. The host (who knows where the car is and will never reveal the car) opens Door 2, revealing a goat.
3. You are offered the option to switch to Door 3.

### The Host's Likelihood Policy
- If the car is behind Door 1: Host opens Door 2 with probability $1/2$ (he can choose Door 2 or 3).
- If the car is behind Door 2: Host CANNOT open Door 2; probability is $0$.
- If the car is behind Door 3: Host is FORCED to open Door 2; probability is $1.0$.

$$
P(C_i \mid O_2) = \frac{P(O_2 \mid C_i) P(C_i)}{\sum_j P(O_2 \mid C_j) P(C_j)}
$$

In [5]:
# 1. Exact Bayesian Matrix Calculation
doors = [1, 2, 3]
prior = np.array([1/3, 1/3, 1/3])

# Player chooses Door 1. Host opens Door 2.
# Likelihood: P(Host opens Door 2 | Car is at Door i)
likelihood = np.array([1/2, 0.0, 1.0])

unnorm_post = prior * likelihood
evidence = unnorm_post.sum() # Marginal probability P(Host opens Door 2) = 1/6 + 0 + 1/3 = 1/2
posterior = unnorm_post / evidence

print("=== Exact Bayesian Updating for Monty Hall ===")
for d, pri, lik, post in zip(doors, prior, likelihood, posterior):
    print(f"Door {d}: Prior = {pri:.4f} | Likelihood = {lik:.4f} | Posterior = {post:.4f} ({post*100:.1f}%)")

# 2. Monte Carlo Simulation: 10,000 Independent Trials
n_trials = 10_000
car_doors = np.random.choice(doors, size=n_trials)
player_picks = np.ones(n_trials, dtype=int) # Always pick Door 1 without loss of generality

host_opens = np.zeros(n_trials, dtype=int)
for i in range(n_trials):
    car = car_doors[i]
    eligible = [d for d in doors if d != 1 and d != car]
    host_opens[i] = np.random.choice(eligible)

stay_wins = (player_picks == car_doors)
switch_picks = np.zeros(n_trials, dtype=int)
for i in range(n_trials):
    remaining = [d for d in doors if d != 1 and d != host_opens[i]][0]
    switch_picks[i] = remaining

switch_wins = (switch_picks == car_doors)

print(f"\n=== Monte Carlo Simulation Results ({n_trials:,} Games) ===")
print(f"Win Rate by STAYING:  {stay_wins.mean():.4f} ({stay_wins.mean()*100:.1f}%) [Exact Theory: 33.3%]")
print(f"Win Rate by SWITCHING: {switch_wins.mean():.4f} ({switch_wins.mean()*100:.1f}%) [Exact Theory: 66.7%]")

=== Exact Bayesian Updating for Monty Hall ===
Door 1: Prior = 0.3333 | Likelihood = 0.5000 | Posterior = 0.3333 (33.3%)
Door 2: Prior = 0.3333 | Likelihood = 0.0000 | Posterior = 0.0000 (0.0%)
Door 3: Prior = 0.3333 | Likelihood = 1.0000 | Posterior = 0.6667 (66.7%)

=== Monte Carlo Simulation Results (10,000 Games) ===
Win Rate by STAYING:  0.3354 (33.5%) [Exact Theory: 33.3%]
Win Rate by SWITCHING: 0.6646 (66.5%) [Exact Theory: 66.7%]


## ▶ RUN TOGETHER: 4. Likelihood vs. Probability Density: The Duality in Code

A foundational concept in Bayesian statistics: **A likelihood function is NOT a probability distribution**.
1. **$P(D \mid \theta)$ (Probability Mass / Density)**: Evaluated over possible *data outcomes* $k \in \{0, \dots, N\}$ holding parameter $\theta$ fixed. **Sums strictly to 1.0**.
2. **$\mathcal{L}(\theta \mid D)$ (Likelihood Function)**: Evaluated over candidate *parameter values* $\theta \in [0, 1]$ holding observed data $D$ fixed. **Does not sum or integrate to 1.0** (area under the Binomial likelihood curve equals $\frac{1}{N+1}$).
3. **$p(\theta \mid D)$ (Posterior Density)**: Combines prior and likelihood, normalized by marginal evidence. **Integrates strictly to 1.0**.

In [6]:
N = 10
observed_k = 7
theta_fixed = 0.70

# 1. Forward Generative World: Binomial PMF over data space k in {0, ..., 10}
k_grid = np.arange(0, N + 1)
pmf_data = stats.binom.pmf(k_grid, N, theta_fixed)
print(f"1. Sum of PMF over all data outcomes k: {pmf_data.sum():.4f} (Strictly 1.0!)")

# 2. Inverse Inferential World: Likelihood over parameter space theta in [0, 1]
theta_grid = np.linspace(0.001, 0.999, 400)
lik_curve = stats.binom.pmf(observed_k, N, theta_grid)

# Flat prior Beta(1,1) updates to Beta(1+7, 1+3) = Beta(8, 4)
post_curve = stats.beta.pdf(theta_grid, 1 + observed_k, 1 + N - observed_k)

# Integrate both curves numerically using trapezoidal rule
area_lik = integrate.trapezoid(lik_curve, theta_grid)
area_post = integrate.trapezoid(post_curve, theta_grid)

print(f"2. Area under Likelihood curve L(theta | k=7): {area_lik:.4f} (Equals 1/(N+1) = 1/11 = {1/(N+1):.4f}, NOT 1.0!)")
print(f"3. Area under Posterior density p(theta | k=7): {area_post:.4f} (Strictly 1.0!)")

1. Sum of PMF over all data outcomes k: 1.0000 (Strictly 1.0!)
2. Area under Likelihood curve L(theta | k=7): 0.0909 (Equals 1/(N+1) = 1/11 = 0.0909, NOT 1.0!)
3. Area under Posterior density p(theta | k=7): 1.0000 (Strictly 1.0!)


In [7]:
# Interactive Dual-Panel Comparison
fig_dual = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        '(a) Probability Mass P(D | θ = 0.70)<br>Sums strictly to 1.0 over data k',
        '(b) Likelihood vs. Posterior Density<br>Integrates over parameter θ (Fixed k = 7)'
    ]
)

bar_colors = ['#cbd5e1'] * (N + 1)
bar_colors[observed_k] = '#ea580c'
fig_dual.add_trace(go.Bar(x=k_grid, y=pmf_data, marker_color=bar_colors, name='PMF P(k | θ=0.70)'), row=1, col=1)

fig_dual.add_trace(go.Scatter(
    x=theta_grid, y=lik_curve, mode='lines',
    line=dict(color='#ea580c', width=2, dash='dash'),
    name=f'Likelihood L(θ | k=7) [Area = {area_lik:.3f}]'
), row=1, col=2)

fig_dual.add_trace(go.Scatter(
    x=theta_grid, y=post_curve, mode='lines',
    line=dict(color='#2563eb', width=2.5),
    name=f'Posterior p(θ | k=7) [Area = {area_post:.3f}]'
), row=1, col=2)

fig_dual.update_layout(template='plotly_white', height=420, showlegend=True)
fig_dual.show()

## ▶ RUN TOGETHER: 5. Sequential Bernoulli Updating & Order Invariance

Bayesian learning is naturally cumulative: **yesterday's posterior becomes today's prior**.

Under conditionally independent Bernoulli trials with a conjugate $\operatorname{Beta}(\alpha, \beta)$ prior:
$$
\operatorname{Beta}(\alpha, \beta) + y_t \implies \operatorname{Beta}(\alpha + y_t, \; \beta + 1 - y_t)
$$
Let us simulate sequential updating for the exact sequence of 5 coin flips from Chapter 2: $[1, 0, 1, 1, 0]$ starting from prior $\operatorname{Beta}(2, 2)$:

In [8]:
flips = [1, 0, 1, 1, 0] # Heads=1, Tails=0
alpha_0, beta_0 = 2, 2

history = [{
    'step': 0, 'observation': 'Prior',
    'alpha': alpha_0, 'beta': beta_0,
    'mean': alpha_0 / (alpha_0 + beta_0),
    'variance': (alpha_0 * beta_0) / (((alpha_0 + beta_0)**2) * (alpha_0 + beta_0 + 1))
}]

curr_a, curr_b = alpha_0, beta_0
for t, y in enumerate(flips, start=1):
    curr_a += y
    curr_b += (1 - y)
    history.append({
        'step': t,
        'observation': f'Flip {t}: {"Heads" if y == 1 else "Tails"}',
        'alpha': curr_a,
        'beta': curr_b,
        'mean': curr_a / (curr_a + curr_b),
        'variance': (curr_a * curr_b) / (((curr_a + curr_b)**2) * (curr_a + curr_b + 1))
    })

df_seq = pd.DataFrame(history)
print("=== Step-by-Step Sequential Updating Table ===")
print(df_seq[['step', 'observation', 'alpha', 'beta', 'mean', 'variance']].to_string(index=False))

# Confirm order invariance: Updating on batch (3 Heads, 2 Tails)
batch_a = alpha_0 + sum(flips)
batch_b = beta_0 + (len(flips) - sum(flips))
print(f"\nAll-at-once update: Beta({batch_a}, {batch_b}) with mean = {batch_a/(batch_a+batch_b):.4f}")
print("Does sequential update equal all-at-once update?", curr_a == batch_a and curr_b == batch_b)

=== Step-by-Step Sequential Updating Table ===
 step   observation  alpha  beta     mean  variance
    0         Prior      2     2 0.500000  0.050000
    1 Flip 1: Heads      3     2 0.600000  0.040000
    2 Flip 2: Tails      3     3 0.500000  0.035714
    3 Flip 3: Heads      4     3 0.571429  0.030612
    4 Flip 4: Heads      5     3 0.625000  0.026042
    5 Flip 5: Tails      5     4 0.555556  0.024691

All-at-once update: Beta(5, 4) with mean = 0.5556
Does sequential update equal all-at-once update? True


## 🛑 STOP 2: Verify Order Invariance
`✍ WRITE`:
1. Does updating on $[1, 0, 1, 1, 0]$ produce the exact same final posterior as updating on the reordered sequence $[0, 0, 1, 1, 1]$? Explain why algebraically from the likelihood product.
2. In real-world data collection (e.g. longitudinal cognitive testing or public opinion polls across election cycles), why might order invariance fail?

---
# Part II: Optional Homework & Specialist Extensions

## 🏠 OPTIONAL HOMEWORK: 6. Precision-Weighted Information Aggregation (ROS Ch. 9)
Gelman, Hill, and Vehtari (*Regression and Other Stories*, Ch. 9) demonstrate Bayesian updating via precision weighting:
1. **Model-based Prior**: Economic forecast of Democratic vote share $\mu_0 = 0.524$ with $\sigma_0 = 0.041$.
2. **Survey Evidence**: $n = 400$ polled voters, $y = 190$ supporting Democrat $\implies \bar{y} = 190/400 = 0.475$, with sampling standard error $\sigma_{\text{survey}} = \sqrt{\bar{y}(1-\bar{y})/n} \approx 0.02497$.

Under Normal updating, **precisions** $\tau = 1/\sigma^2$ add:
$$
\tau_{\text{post}} = \tau_0 + \tau_{\text{data}}, \qquad \mu_{\text{post}} = \frac{\tau_0 \mu_0 + \tau_{\text{data}} \bar{y}}{\tau_{\text{post}}}, \qquad \sigma_{\text{post}} = \frac{1}{\sqrt{\tau_{\text{post}}}}
$$

In [9]:
mu_0 = 0.524
se_0 = 0.041

n_survey = 400
y_survey = 190
y_bar = y_survey / n_survey
se_survey = np.sqrt(y_bar * (1 - y_bar) / n_survey)

tau_0 = 1.0 / (se_0 ** 2)
tau_data = 1.0 / (se_survey ** 2)
tau_post = tau_0 + tau_data

mu_post = (tau_0 * mu_0 + tau_data * y_bar) / tau_post
se_post = 1.0 / np.sqrt(tau_post)

print("=== ROS Ch. 9: Normal Precision Weighting Results ===")
print(f"Prior:     Mean = {mu_0:.4f}, SE = {se_0:.4f} (Weight: {tau_0 / tau_post * 100:.1f}%)")
print(f"Survey:    Mean = {y_bar:.4f}, SE = {se_survey:.4f} (Weight: {tau_data / tau_post * 100:.1f}%)")
print(f"Posterior: Mean = {mu_post:.4f}, SE = {se_post:.4f}")
print(f"\nNotice that posterior SE ({se_post:.4f}) is strictly smaller than both prior ({se_0:.4f}) and survey ({se_survey:.4f})!")

=== ROS Ch. 9: Normal Precision Weighting Results ===
Prior:     Mean = 0.5240, SE = 0.0410 (Weight: 27.1%)
Survey:    Mean = 0.4750, SE = 0.0250 (Weight: 72.9%)
Posterior: Mean = 0.4883, SE = 0.0213

Notice that posterior SE (0.0213) is strictly smaller than both prior (0.0410) and survey (0.0250)!


## 🏠 OPTIONAL HOMEWORK: 7. When Updating Fails: The Exchangeability Assumption (ROS Ch. 4)
Naive Bayesian updating assumes that observations are **exchangeable** — meaning their labels or collection dates do not carry independent predictive information about a drifting parameter.

If public opinion shifts over time (as in political polling across different decades), aggregating older polls with new ones as if they were drawn from the same stationary distribution leads to severe overconfidence in an obsolete parameter.

Consider the ROS historical death penalty polling series (Gelman et al., Ch. 4):

In [10]:
# Historical polling series demonstration (ROS Ch. 4)
polls_data = pd.DataFrame([
    {'year': 1953, 'support': 0.68, 'sample_size': 1000},
    {'year': 1966, 'support': 0.42, 'sample_size': 1000},
    {'year': 1976, 'support': 0.66, 'sample_size': 1000},
    {'year': 1994, 'support': 0.80, 'sample_size': 1000},
    {'year': 2016, 'support': 0.53, 'sample_size': 1000}
])

print("=== Historical Gallup Polls on Death Penalty Support ===")
print(polls_data.to_string(index=False))

print("""
If an analyst naively pooled 1966 (42% support) and 1994 (80% support) via conjugate updating:
- The pooled sample size would be N = 2,000.
- The standard error would shrink to ~0.01.
- But the estimated mean (~61%) describes NEITHER 1966 NOR 1994!
Exchangeability was violated because the true parameter theta_t changed over time.
""")

=== Historical Gallup Polls on Death Penalty Support ===
 year  support  sample_size
 1953     0.68         1000
 1966     0.42         1000
 1976     0.66         1000
 1994     0.80         1000
 2016     0.53         1000

If an analyst naively pooled 1966 (42% support) and 1994 (80% support) via conjugate updating:
- The pooled sample size would be N = 2,000.
- The standard error would shrink to ~0.01.
- But the estimated mean (~61%) describes NEITHER 1966 NOR 1994!
Exchangeability was violated because the true parameter theta_t changed over time.



---
# Part III: Exit Record (Research Protocol Milestone 2)

`✍ WRITE`: Record your answers to these three reflection questions in your research notebook:
1. **The Reference Group**: Identify an example in your own research domain where conditioning on the wrong denominator produces a distorted inference (similar to Berkeley admissions).
2. **Prior vs. Data Balance**: In your study, will prior knowledge carry more precision than new empirical data, or will new observations dominate the posterior?
3. **Exchangeability Audit**: Name one factor (e.g. repeated testing, secular time trends, differing experimental sites) that could violate exchangeability in your dataset.

In [11]:
# ==============================================================================
# ✅ Reproducibility Footer
# ==============================================================================
print(f"Timestamp (UTC): {datetime.datetime.now(datetime.timezone.utc).isoformat()}")
print(f"Random seed:     42")
print(f"Python version:  {sys.version.split()[0]}")
print(f"NumPy:           {np.__version__}")
print(f"SciPy:           {stats.__version__ if hasattr(stats, '__version__') else 'installed'}")
print(f"Pandas:          {pd.__version__}")
print(f"Plotly:          {go.__version__ if hasattr(go, '__version__') else 'installed'}")

Timestamp (UTC): 2026-09-12T10:41:12.078330+00:00
Random seed:     42
Python version:  3.12.13
NumPy:           2.4.3
SciPy:           installed
Pandas:          3.0.2
Plotly:          installed
